# Data Cleaning

This notebook contains all data preparation logic for the Phase 3 analysis.
It loads the raw business and NOAA weather files, cleans and merges them,
and exports two analysis-ready CSV files:

- `csvs/analysis_data.csv` - identified-material monthly sales with weather, 2014-present
- `csvs/weather_monthly.csv` - national monthly temperature and precipitation, identified-material sales date range

[View this notebook on GitHub](https://github.com/bobbybotbop/INFO2950/blob/main/data_cleaning.ipynb)


## 1. Setup

We import Python libraries used for cleaning, analyzing, and visualizing the data.
All source files are in the `csvs/` folder.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from IPython.display import display

plt.rcParams['figure.figsize'] = (10, 4)


## 2. Business Data

### 2.1 Data sources

We use three files from the business accounting software and web storefront:

| File | Use |
| --- | --- |
| `salesByCustomerDetail.CSV` | Invoice lines used to calculate monthly sales. |
| `itemData.CSV` | Product descriptions and catalog types used to identify hats. |
| `inventoryDataWithMoreDetail.csv` | Web-store product catalog with HTML `material:` fields. |

### 2.2 Load the business files

We read all columns as text initially so that product IDs are not accidentally converted.


In [2]:
sales_raw = pd.read_csv('csvs/salesByCustomerDetail.CSV', encoding='cp1252', dtype=str).fillna('')
catalog = pd.read_csv('csvs/itemData.CSV', encoding='cp1252', dtype=str).fillna('')
inventory_detail = pd.read_csv('csvs/inventoryDataWithMoreDetail.csv', encoding='utf-8-sig', dtype=str).fillna('')

print('Sales report rows:', len(sales_raw))
print('Catalog rows:', len(catalog))
print('Inventory detail rows:', len(inventory_detail))
print('Sales row types (blank = report heading or subtotal):')
display(sales_raw['Type'].value_counts())


Sales report rows: 32768
Catalog rows: 7866
Inventory detail rows: 931
Sales row types (blank = report heading or subtotal):


Type
Invoice    31877
             891
Name: count, dtype: int64

### 2.3 Clean transactions and match products

We filter to invoice rows, convert dates and numeric columns, extract product IDs,
and join product descriptions from the catalog. Both exports contain exactly 32,768 rows,
suggesting a possible export limit; we retain all rows and note this limitation.


In [3]:
sales = sales_raw[sales_raw['Type'] == 'Invoice'].copy()
sales['source_csv_row'] = sales.index + 2
sales['date'] = pd.to_datetime(sales['Date'], format='%m/%d/%Y', errors='coerce')

for column in ['Qty', 'Amount', 'Sales Price']:
    sales[column] = sales[column].str.replace(',', '', regex=False)
    sales[column] = sales[column].str.replace('$', '', regex=False)
    sales[column] = sales[column].str.replace('(', '-', regex=False)
    sales[column] = sales[column].str.replace(')', '', regex=False)
    sales[column] = pd.to_numeric(sales[column], errors='coerce')
sales = sales.rename(columns={'Qty': 'quantity', 'Amount': 'amount', 'Sales Price': 'unit_price'})

sales['item_id'] = sales['Item'].str.split(' (', n=1, regex=False).str[0].str.strip()
catalog['item_id'] = catalog['Item'].str.strip()
products = catalog[['item_id', 'Type', 'Description']].copy()
products = products.rename(columns={'Type': 'catalog_type', 'Description': 'catalog_description'})
sales = sales.merge(products, on='item_id', how='left', validate='many_to_one')

print('Invoice lines:', len(sales))
print('Unmatched catalog lines:', sales['catalog_type'].isna().sum())
print('First invoice date:', sales['date'].min())
print('Last invoice date:', sales['date'].max())
print('Missing numeric values:')
display(sales[['quantity', 'amount', 'unit_price']].isna().sum())
print('Zero or negative quantity lines:', (sales['quantity'] <= 0).sum())


Invoice lines: 31877
Unmatched catalog lines: 1
First invoice date: 2006-10-01 00:00:00
Last invoice date: 2026-08-13 00:00:00
Missing numeric values:


quantity         0
amount         466
unit_price    1089
dtype: int64

Zero or negative quantity lines: 422


### 2.4 Identify hats and classify materials

We identify hat products using keywords (hat, cap, fedora, etc.) restricted to merchandise
catalog types. We classify materials using HTML `material:` fields from the web-store
catalog, falling back to keyword matching. Products that cannot be classified remain `unknown`.


In [4]:
def extract_material(desc):
    m = re.search(r'material:\s*([^<\n]+)', desc, re.IGNORECASE)
    if not m:
        return 'unknown'
    raw = re.sub(r'&[a-z#0-9]+;', ' ', m.group(1)).strip().lower()
    found = []
    if 'wool felt' in raw:
        found.append('wool')
    else:
        if 'wool' in raw:
            found.append('wool')
        if 'felt' in raw:
            found.append('felt')
    if 'cotton' in raw:
        found.append('cotton')
    if 'acrylic' in raw:
        found.append('acrylic')
    if 'polyester' in raw:
        found.append('polyester')
    if 'leather' in raw:
        found.append('leather')
    if 'straw' in raw or 'paper' in raw or 'raffia' in raw:
        found.append('straw')
    if not found:
        return 'unknown'
    if len(found) == 1:
        return found[0]
    return 'mixed_or_ambiguous'

detail = inventory_detail.copy()
detail['item_id'] = (detail['Product Code']
                     .str.strip()
                     .str.replace(r'\s+same as.*$', '', case=False, regex=True))
detail = detail.drop_duplicates('item_id')
detail['material_detail'] = detail['Description'].apply(extract_material)
material_lookup = detail.set_index('item_id')['material_detail']

sales['description_text'] = sales['catalog_description'].fillna('') + ' ' + sales['Item']
sales['description_text'] = sales['description_text'].str.lower()
sales['words'] = sales['description_text'].str.replace(r'[^a-z]+', ' ', regex=True)
sales['words'] = ' ' + sales['words'] + ' '

hat_words = ['hat', 'hats', 'cap', 'caps', 'ivy', 'ivies', 'fedora', 'fedoras',
             'beanie', 'beanies', 'beret', 'berets', 'cloche', 'cloches',
             'fascinator', 'fascinators', 'visor', 'visors', 'bonnet', 'bonnets',
             'duckbill', 'newsboy', 'trilby', 'floppy', 'panama']
sales['candidate_hat'] = False
for word in hat_words:
    matches = sales['words'].str.contains(' ' + word + ' ', regex=False)
    sales.loc[matches, 'candidate_hat'] = True

merchandise_types = ['Inventory Part', 'Inventory Assembly', 'Non-inventory Part']
sales['candidate_hat'] = sales['candidate_hat'] & sales['catalog_type'].isin(merchandise_types)

sales['material_kw'] = 'unknown'
kw_matches = pd.Series(0, index=sales.index)
for mat in ['wool', 'straw', 'cotton', 'polyester', 'acrylic', 'leather', 'felt']:
    m = sales['words'].str.contains(' ' + mat + ' ', regex=False)
    if mat == 'straw':
        m = m | sales['words'].str.contains(' raffia ', regex=False)
    sales.loc[m, 'material_kw'] = mat
    kw_matches += m.astype(int)
sales.loc[kw_matches > 1, 'material_kw'] = 'mixed_or_ambiguous'

sales['material_detail'] = sales['item_id'].map(material_lookup).fillna('unknown')
sales['material'] = sales['material_detail'].where(
    sales['material_detail'] != 'unknown',
    sales['material_kw']
)

keep = sales['candidate_hat'] & sales['date'].notna() & (sales['quantity'] > 0)
clean_sales = sales.loc[keep, ['source_csv_row', 'date', 'item_id', 'quantity',
                               'amount', 'unit_price', 'material']].copy()
clean_sales['month'] = clean_sales['date'].dt.to_period('M').dt.to_timestamp()
print('Retained candidate-hat invoice lines:', len(clean_sales))
display(clean_sales.groupby('material')['quantity'].sum().sort_values(ascending=False))


Retained candidate-hat invoice lines: 23072


material
unknown               666287
cotton                 32637
wool                   22615
felt                   12318
straw                  10978
acrylic                 7408
mixed_or_ambiguous      7085
leather                 2529
polyester               2514
Name: quantity, dtype: int64

### 2.5 Monthly sales aggregation

We group transactions by month and material, summing quantities. We restrict to rows with
an identified material (excluding `unknown` and `mixed_or_ambiguous`), since 88.1% of
quantity lacks material information. Months without identified-material sales are left
missing rather than set to zero, because we cannot distinguish zero sales from gaps.


In [5]:
monthly_sales = (
    clean_sales[~clean_sales['material'].isin(['unknown', 'mixed_or_ambiguous'])]
    .groupby(['month', 'material'])['quantity'].sum()
    .reset_index()
    .rename(columns={'quantity': 'recorded_quantity'})
)

month_range = pd.date_range(clean_sales['month'].min(), clean_sales['month'].max(), freq='MS')
observed_month_count = clean_sales['month'].nunique()
print('Observed sales months:', observed_month_count)
print('Months without retained observations:', len(month_range) - observed_month_count)
print('Known-material month-material rows:', len(monthly_sales))
display(monthly_sales.head())


Observed sales months: 234
Months without retained observations: 5
Known-material month-material rows: 649


,month,material,recorded_quantity
0,2007-05-01,cotton,144
1,2007-06-01,cotton,108
2,2007-07-01,cotton,192
3,2007-08-01,cotton,120
4,2007-08-01,felt,60


## 3. Weather Data

We use NOAA monthly temperature and precipitation records for the contiguous United States.
The files have one row per year with columns for each month; we select the national
(region 110, division 0) temperature and precipitation series.

Record format: https://www.ncei.noaa.gov/monitoring-content/data/us/climdiv/monthly/current/state-readme.txt


In [6]:
month_columns = ['01', '02', '03', '04', '05', '06',
                 '07', '08', '09', '10', '11', '12']
column_names = ['code'] + month_columns

temperature_raw = pd.read_csv('csvs/TempreatureData.CSV', sep=r'\s+',
                              header=None, names=column_names, dtype={'code': str})
precipitation_raw = pd.read_csv('csvs/PrecipitationData.CSV', sep=r'\s+',
                                header=None, names=column_names, dtype={'code': str})

temperature = temperature_raw[temperature_raw['code'].str[:6] == '110002'].copy()
precipitation = precipitation_raw[precipitation_raw['code'].str[:6] == '110001'].copy()
temperature['year'] = temperature['code'].str[6:10]
precipitation['year'] = precipitation['code'].str[6:10]
print('National temperature years:', len(temperature))
print('National precipitation years:', len(precipitation))
display(temperature[['year'] + month_columns].tail(2))


National temperature years: 132
National precipitation years: 132


,year,01,02,03,04,05,06,07,08,09,10,11,12
7792,2025,29.16,34.74,46.83,53.56,61.70,71.20,75.42,73.38,67.95,56.95,46.69,37.83
7793,2026,33.12,40.26,50.85,54.77,61.68,70.66,76.91,75.54,69.42,-99.90,-99.90,-99.90


### 3.1 Clean temperature data

We reshape from wide format (one row per year) to long format (one row per month).
We replace the NOAA missing-value code (-99.90) with NaN and convert from Fahrenheit to Celsius.


In [7]:
temperature_monthly = temperature.melt(id_vars='year', value_vars=month_columns,
                                       var_name='month_number', value_name='temperature_f')
temperature_monthly['temperature_f'] = temperature_monthly['temperature_f'].replace(-99.90, float('nan'))
temperature_monthly['month'] = pd.to_datetime(
    temperature_monthly['year'] + '-' + temperature_monthly['month_number'] + '-01')
temperature_monthly['temperature_c'] = (temperature_monthly['temperature_f'] - 32) * 5 / 9
temperature_monthly = temperature_monthly[['month', 'temperature_c']].sort_values('month')
print('Missing national temperature months:', temperature_monthly['temperature_c'].isna().sum())
display(temperature_monthly.tail())


Missing national temperature months: 3


,month,temperature_c
1055,2026-08-01,24.188889
1187,2026-09-01,20.788889
1319,2026-10-01,NaN
1451,2026-11-01,NaN
1583,2026-12-01,NaN


### 3.2 Clean precipitation data

Same reshape and missing-value treatment for precipitation; we convert inches to millimeters.


In [8]:
precipitation_monthly = precipitation.melt(id_vars='year', value_vars=month_columns,
                                           var_name='month_number', value_name='precipitation_in')
precipitation_monthly['precipitation_in'] = precipitation_monthly['precipitation_in'].replace(-9.99, float('nan'))
precipitation_monthly['month'] = pd.to_datetime(
    precipitation_monthly['year'] + '-' + precipitation_monthly['month_number'] + '-01')
precipitation_monthly['precipitation_mm'] = precipitation_monthly['precipitation_in'] * 25.4
precipitation_monthly = precipitation_monthly[['month', 'precipitation_mm']].sort_values('month')
print('Missing national precipitation months:', precipitation_monthly['precipitation_mm'].isna().sum())
display(precipitation_monthly.tail())


Missing national precipitation months: 3


,month,precipitation_mm
1055,2026-08-01,58.674
1187,2026-09-01,68.326
1319,2026-10-01,NaN
1451,2026-11-01,NaN
1583,2026-12-01,NaN


## 4. Merge Sales and Weather

We combine temperature and precipitation into a single monthly weather table filtered
to the sales date range, then join with the identified-material monthly sales.


In [9]:
weather_monthly = temperature_monthly.merge(precipitation_monthly, on='month',
                                            how='outer', validate='one_to_one')
weather_monthly = weather_monthly.sort_values('month')

start_month = monthly_sales['month'].min()
end_month = monthly_sales['month'].max()
weather_monthly = weather_monthly[weather_monthly['month'].between(start_month, end_month)].copy()

analysis_data = monthly_sales.merge(weather_monthly, on='month', how='left', validate='many_to_one')
print('Sales month-material rows before merge:', len(monthly_sales))
print('Rows after merge:', len(analysis_data))
print('Missing weather values in merged data:')
display(analysis_data[['temperature_c', 'precipitation_mm']].isna().sum())
display(analysis_data.head())


Sales month-material rows before merge: 649
Rows after merge: 649
Missing weather values in merged data:


temperature_c       0
precipitation_mm    0
dtype: int64

,month,material,recorded_quantity,temperature_c,precipitation_mm
0,2007-05-01,cotton,144,16.838889,69.850
1,2007-06-01,cotton,108,21.027778,73.914
2,2007-07-01,cotton,192,23.711111,73.152
3,2007-08-01,cotton,120,23.677778,66.040
4,2007-08-01,felt,60,23.677778,66.040


## 5. Material Coverage and the 2014 Analysis Period

We calculate each year's identified-material fraction to determine a defensible
regression start year.


In [10]:
clean_sales['material_known'] = ~clean_sales['material'].isin(['unknown', 'mixed_or_ambiguous'])
clean_sales['year'] = clean_sales['date'].dt.year

annual = clean_sales.groupby('year').agg(
    total_qty=('quantity', 'sum'),
    known_qty=('quantity', lambda x: x[clean_sales.loc[x.index, 'material_known']].sum())
).assign(known_pct=lambda df: df['known_qty'] / df['total_qty'] * 100).round(1)

print('Annual known-material coverage:')
display(annual[['total_qty', 'known_qty', 'known_pct']])

threshold_year = annual[annual['known_pct'] >= 10].index.min()
print(f'First year where known material >= 10% of annual quantity: {threshold_year:.0f}')


Annual known-material coverage:


,total_qty,known_qty,known_pct
year,,,
2006,63152,0,0.0
2007,88515,708,0.8
2008,20277,27,0.1
2009,10876,990,9.1
2010,29474,1312,4.5
2011,132475,2212,1.7
2012,67020,1952,2.9
2013,59184,1894,3.2
2014,52042,8271,15.9


First year where known material >= 10% of annual quantity: 2014


Known-material quantity first reaches 10% of annual total in **2014**. All regression
analyses in `phase3.ipynb` are therefore restricted to January 2014 onward.

The overall unknown-material share across the full dataset is 88.1%, reflecting a historical
data-quality gap: the business did not document product materials before it moved online,
and many of those older products are no longer sold.


## 6. Export Analysis-Ready CSVs

We export two CSV files for use in `phase3.ipynb`:

- **`analysis_data.csv`**: one row per identified material per month, January 2014 onward.
- **`weather_monthly.csv`**: one row per month for the identified-material sales date range (May 2007-Aug 2026).


In [11]:
analysis_export = analysis_data[analysis_data['month'] >= '2014-01-01'].copy()
analysis_export['month'] = analysis_export['month'].dt.strftime('%Y-%m-%d')
analysis_export[['month', 'material', 'recorded_quantity',
                 'temperature_c', 'precipitation_mm']].to_csv('csvs/analysis_data.csv', index=False)

weather_export = weather_monthly.copy()
weather_export['month'] = weather_export['month'].dt.strftime('%Y-%m-%d')
weather_export[['month', 'temperature_c', 'precipitation_mm']].to_csv('csvs/weather_monthly.csv', index=False)

print('Exported analysis_data.csv:', len(analysis_export), 'rows')
print('Exported weather_monthly.csv:', len(weather_export), 'rows')


Exported analysis_data.csv: 567 rows
Exported weather_monthly.csv: 232 rows


In [12]:
ad = pd.read_csv('csvs/analysis_data.csv', parse_dates=['month'])
wm = pd.read_csv('csvs/weather_monthly.csv', parse_dates=['month'])

print('analysis_data.csv shape:', ad.shape)
print('columns:', list(ad.columns))
print('date range:', ad['month'].min().strftime('%Y-%m'), 'to', ad['month'].max().strftime('%Y-%m'))
print('materials:', sorted(ad['material'].unique()))
print()
print('weather_monthly.csv shape:', wm.shape)
print('columns:', list(wm.columns))
print('date range:', wm['month'].min().strftime('%Y-%m'), 'to', wm['month'].max().strftime('%Y-%m'))


analysis_data.csv shape: (567, 5)
columns: ['month', 'material', 'recorded_quantity', 'temperature_c', 'precipitation_mm']
date range: 2014-01 to 2026-08
materials: ['acrylic', 'cotton', 'felt', 'leather', 'polyester', 'straw', 'wool']

weather_monthly.csv shape: (232, 3)
columns: ['month', 'temperature_c', 'precipitation_mm']
date range: 2007-05 to 2026-08
